In [ ]:
con.close()

In [ ]:
import os
from dataclasses import dataclass
from datetime import date

import duckdb
import polars as pl
from pydantic_ai import (
    Agent,
    FunctionToolCallEvent,
    FunctionToolResultEvent,
    ModelRetry,
    PartDeltaEvent,
    PartStartEvent,
    RunContext,
    TextPartDelta,
    ThinkingPart,
    ThinkingPartDelta,
)
from pydantic_ai.models.openrouter import OpenRouterModel
from pydantic_ai.providers.openrouter import OpenRouterProvider
from sentence_transformers import SentenceTransformer

open_router_key = os.getenv("OPEN_ROUTER_API_KEY")
TABLE_NAME = "int_supervalu"

bi_encoder = SentenceTransformer("microsoft/harrier-oss-v1-270m", trust_remote_code=True)

con = duckdb.connect("/Users/brianbarry/supermarket_scraping/supermarket_data.db")
con.sql("CREATE OR REPLACE SECRET secret (TYPE s3,PROVIDER credential_chain);")
con.install_extension("vss", repository="core")
con.load_extension("vss")
con.sql("set hnsw_enable_experimental_persistence = True")

# con.sql("PRAGMA create_fts_index('int_supervalu','id','title_cleaned','ingredients', overwrite=1)")

df_sv = con.sql("SELECT * FROM int_supervalu").pl()

In [ ]:
@dataclass
class Deps:
    conn: duckdb.DuckDBPyConnection


def get_schema(conn: duckdb.DuckDBPyConnection, table: str) -> str:
    rows = conn.execute(
        """
        SELECT column_name, data_type
        FROM information_schema.columns
        WHERE table_name = ?
        ORDER BY ordinal_position
        """,
        [table],
    ).fetchall()
    cols = ",\n  ".join(f"{name} {dtype}" for name, dtype in rows)
    return f"CREATE TABLE {table} (\n  {cols}\n);"


model = OpenRouterModel(
    "xiaomi/mimo-v2.5",
    provider=OpenRouterProvider(api_key=open_router_key),
)
agent = Agent(model, deps_type=Deps)


@agent.system_prompt
def system_prompt(ctx: RunContext[Deps]) -> str:
    return f"""\
You are a data analyst. Answer the user's question about the `{TABLE_NAME}`
table by writing DuckDB SQL and running it with the `run_sql` tool.
Issue only SELECT queries, and inspect the returned rows before giving your
final answer.

Schema:

{get_schema(ctx.deps.conn, TABLE_NAME)}

Today's date is {date.today()}.
"""


@agent.tool()
def run_sql(ctx: RunContext[Deps], sql: str) -> str:
    """Run a read-only DuckDB SELECT query and return up to 20 result rows.

    Args:
        ctx: run context
        sql: a single DuckDB SELECT statement
    """
    if not (sql.lstrip().upper().startswith("SELECT") or sql.lstrip().upper().startswith("WITH")):
        raise ModelRetry("Only SELECT queries are allowed.")
    try:
        rel = ctx.deps.conn.sql(sql)
        columns = rel.columns
        rows = rel.fetchmany(25)
    except duckdb.Error as e:
        raise ModelRetry(f"Query failed: {e}") from e

    if not rows:
        return "(no rows)"
    header = " | ".join(columns)
    body = "\n".join(" | ".join(str(v) for v in row) for row in rows)
    return f"{header}\n{body}"

@agent.tool()
def hybrid_search(ctx: RunContext[Deps], search_term: str, embed_weight: float = 0.6, fts_weight: float = 0.4) -> str:
    """Run hybrid search with reciprocal rank fusion using user query on DuckDB table. Hybrid search comprises bm25 index title_clean and ingredients
       and hnsw index on title_clean_embeddings. Run this tool initially to get top searches relevant to the users query and decide the relevant filters
       to answer the initial query. Ths function returns the title, ingredients, description and supermarket categories which should provide relevant
       information for later queries. With the results of this ouput, you should choose the relevant filters and use these for the follow up queries when necessary
       to make searches more efficient

        Args:
            ctx: run context
            search_term: initil user search
            embed_weight: weight of embedding rank in rrf score, make this higher when query is fuzzy and requires more semantic focus
            fts_weight: bm25 score rank
        """
    search_term_embedding = bi_encoder.encode(search_term, normalize_embeddings=True).tolist()
    query = f"""
    WITH emb AS (
        SELECT
            id, title,
            array_cosine_similarity(title_embedding,{search_term_embedding}::FLOAT[640])::decimal(3,2) AS cosine_distance_score
        FROM int_product_title_embeddings
        WHERE supermarket = 'supervalu'
    ),

    fts AS (
        SELECT
            id, title, ingredients, item_description, category_1, category_2, category_3, category_4,
            fts_main_int_supervalu.match_bm25(id,'{search_term}')::decimal AS bm25_score
        FROM int_supervalu
    ),

    normalized_scores AS (
        SELECT
            fts.id, fts.title, fts.category_1, fts.category_2, fts.category_3, fts.category_4,
            emb.cosine_distance_score AS raw_embed_score,
            fts.bm25_score AS raw_fts_score,
            (fts.bm25_score / (SELECT MAX(bm25_score) FROM fts)) AS norm_fts_score,
            ((emb.cosine_distance_score + 1) / (SELECT MAX(cosine_distance_score) + 1 FROM emb)) AS norm_embd_score,
            ROW_NUMBER() OVER (
                ORDER BY CASE WHEN fts.bm25_score IS NULL THEN 1 ELSE 0 END,
                         fts.bm25_score DESC
            ) AS fts_rank,
            ROW_NUMBER() OVER (ORDER BY emb.cosine_distance_score DESC) AS emb_rank
        FROM fts
        INNER JOIN emb ON fts.id = emb.id
    )

    SELECT
        *,
        ({embed_weight} * norm_embd_score + {fts_weight} * norm_fts_score) AS score_cc,
        (1.0 / (60 + fts_rank) + 1.0 / (60 + emb_rank)) AS rrf_score
    FROM normalized_scores
    --WHERE norm_fts_score IS NOT NULL
    ORDER BY rrf_score DESC
    """
    try:
        rel = ctx.deps.conn.sql(query)
        columns = rel.columns
        rows = rel.fetchmany(21)
    except duckdb.Error as e:
        raise ModelRetry(f"Query failed: {e}") from e

    if not rows:
        return "(no rows)"
    header = " | ".join(columns)
    body = "\n".join(" | ".join(str(v) for v in row) for row in rows)
    return f"{header}\n{body}"



In [ ]:
async def stream_run(agent: Agent, prompt: str, deps: Deps) -> str:
    async with agent.iter(prompt, deps=deps) as run:
        async for node in run:

            # The model is reasoning, writing its answer, or composing a tool call
            if Agent.is_model_request_node(node):
                async with node.stream(run.ctx) as stream:
                    async for event in stream:
                        if isinstance(event, PartStartEvent):
                            if isinstance(event.part, ThinkingPart):
                                print("\n🧠 thinking: ", end="", flush=True)
                                print(event.part.content or "", end="", flush=True)
                        elif isinstance(event, PartDeltaEvent):
                            if isinstance(event.delta, ThinkingPartDelta):
                                print(event.delta.content_delta or "", end="", flush=True)
                            elif isinstance(event.delta, TextPartDelta):
                                print(event.delta.content_delta or "", end="", flush=True)

            # The agent is actually executing the tool calls
            elif Agent.is_call_tools_node(node):
                async with node.stream(run.ctx) as stream:
                    async for event in stream:
                        if isinstance(event, FunctionToolCallEvent):
                            args = event.part.args_as_dict()
                            if event.part.tool_name == "run_sql":
                                print(f"\n\n🔍 SQL:\n{args.get('sql', '')}\n")
                            else:
                                print(f"\n\n🔧 {event.part.tool_name}({args})\n")
                        elif isinstance(event, FunctionToolResultEvent):
                            print(f"📊 result:\n{event.part.content}\n")

    return run.result.output

In [ ]:
hybrid_search(con, search_term='rib eye steak', embed_weight=0.6, fts_weight=0.4)

In [ ]:
con.sql("""SELECT id, title, item_description,score
FROM (
    SELECT *, fts_main_int_supervalu.match_bm25(
        id,
        'rib eye steak'
    ) AS score
    FROM int_supervalu
) sq
WHERE score IS NOT NULL
ORDER BY SCORE DESC
LIMIT 15;""")

# Pre Agent Functions

In [ ]:
con.close()

In [ ]:
from gliner2 import GLiNER2
from gliner2.inference.schema import Schema as gliner2_Schema
from Levenshtein import ratio

extractor = GLiNER2.from_pretrained("fastino/gliner2-base-v1")

user_query_schema = extractor.create_schema().entities({
    "quantity": {
    "description" : "A numeric quantity accompanying a grocery product title i.e 4 pack, 400g, 2 pieces. A number must be included",
    "dtype" : "list",
    "threshold" : 0.5}})


In [ ]:
def get_entity_if_misspelled(entity:str , canonical_list: List[str]) -> str:
    lev_similarities = [ratio(x.lower(),entity, score_cutoff=0.8) for x in canonical_list]
    if all(sim == 0.0 for sim in lev_similarities):
        return []
    else:
        max_index = lev_similarities.index(max(lev_similarities))
        return canonical_list[max_index]

# Agent

In [ ]:
# DB_SCHEMA = get_schema_string(con,"int_supervalu")
SQL_EXAMPLES = [
    {
        'request': 'What is the average price of a bottle of red wine in supervalu',
        'response': "SELECT AVG(price) FROM int_supervalu WHERE category_3 = 'Red Wine AND unit_qty_normalised = 0.75",
    },
]

deps = Deps(con)

output = await stream_run(
    agent,
    "Are there gluten free breaded chicken options available and if so how much are they",
    deps,
)
print("\n\n✅ FINAL:", output)

In [ ]:
result = await agent.run(
    "Are boneless skinless chicken thighs cheaper than chicken fillets?",
    deps=deps,
)

In [ ]:
30000*(15000*0.09/10e6 + 15000*.18/10e6)

In [ ]:
def hybrid_search(con, search_term: str, embed_weight: float = 0.6, fts_weight: float = 0.4) -> str:
    """Run hybrid search with reciprocal rank fusion using user query on DuckDB table. Hybrid search comprises bm25 index title_clean and ingredients
       and hnsw index on title_clean_embeddings. Run this tool initially to get top searches relevant to the users query and decide the relevant filters
       to answer the initial query. Ths function returns the title, ingredients, description and supermarket categories which should provide relevant
       information for later queries. With the results of this ouput, you should choose the relevant filters and use these for the follow up queries when necessary
       to make searches more efficient

        Args:
            ctx: run context
            search_term: initil user search
            embed_weight: weight of embedding rank in rrf score, make this higher when query is fuzzy and requires more semantic focus
            fts_weight: bm25 score rank
        """
    search_term_embedding = bi_encoder.encode(search_term, normalize_embeddings=True).tolist()
    query = f"""
    WITH emb AS (
        SELECT
            id, title,
            array_cosine_similarity(title_embedding,{search_term_embedding}::FLOAT[640])::decimal(3,2) AS cosine_distance_score
        FROM int_product_title_embeddings
        WHERE supermarket = 'supervalu'
    ),

    fts AS (
        SELECT
            id, title, ingredients, item_description, category_1, category_2, category_3, category_4,
            fts_main_int_supervalu.match_bm25(id,'{search_term}')::decimal AS bm25_score
        FROM int_supervalu
    ),

    normalized_scores AS (
        SELECT
            fts.id, fts.title, fts.category_1, fts.category_2, fts.category_3, fts.category_4,
            emb.cosine_distance_score AS raw_embed_score,
            fts.bm25_score AS raw_fts_score,
            (fts.bm25_score / (SELECT MAX(bm25_score) FROM fts)) AS norm_fts_score,
            ((emb.cosine_distance_score + 1) / (SELECT MAX(cosine_distance_score) + 1 FROM emb)) AS norm_embd_score,
            ROW_NUMBER() OVER (
                ORDER BY CASE WHEN fts.bm25_score IS NULL THEN 1 ELSE 0 END,
                         fts.bm25_score DESC
            ) AS fts_rank,
            ROW_NUMBER() OVER (ORDER BY emb.cosine_distance_score DESC) AS emb_rank
        FROM fts
        INNER JOIN emb ON fts.id = emb.id
    )

    SELECT
        *,
        ({embed_weight} * norm_embd_score + {fts_weight} * norm_fts_score) AS score_cc,
        (1.0 / (60 + fts_rank) + 1.0 / (60 + emb_rank)) AS rrf_score
    FROM normalized_scores
    --WHERE norm_fts_score IS NOT NULL
    ORDER BY rrf_score DESC
    """
    return con.sql(query).limit(1055).pl()

In [ ]:
hybrid_df = hybrid_search(con, "plain chicken breast")

In [ ]:
hybrid_df.filter(pl.col("id")=="1809583000")

In [ ]:
hybrid_df.head(20)